In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import SystemMessage

load_dotenv()

/opt/anaconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
llm=ChatGroq(model="llama-3.1-8b-instant")
parser=StrOutputParser()
prompt=ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant.\n\n"
     "Summary of conversation so far:\n{summary}"),
    ("human","{input}")])
chain=prompt|llm|parser
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Summarize the following conversation in 2-3 sentences. "
     "Keep all important facts like names, goals, locations. "
     "Be concise."),
    ("human", "{conversation}")
])
summary_chain=summary_prompt|llm|parser
summaries={}
def update_summary(session_id,human_msg,ai_msg):
    current_summary=summaries.get(session_id,"no conversation yet")
    conversation = (
    f"Current summary: {current_summary}\n\n"
    f"New exchange:\n"
    f"Human: {human_msg}\n"
    f"AI: {ai_msg}"
    )
    new_summary=summary_chain.invoke({"conversation":conversation})
    summaries[session_id]=new_summary
    return new_summary

In [5]:
def chat(message,session_id="session_1"):
    current_summary=summaries.get(session_id,"no conversation yet")
    response=chain.invoke({
    "input":message,
    "summary":current_summary
    })

In [ ]:
new_summary=update_summary(session_id,message,response)